# Linear Algebra for Machine Learning: Interactive Laboratory

Linear algebra is the foundational mathematical language of machine learning. Data points are vectors, datasets and model weights are matrices, and multi-dimensional representations (such as feature maps and batches of token sequences) are higher-order tensors.

### Table of Contents
1. [Vector Geometry, Dot Products & Cosine Distance](#1-vector-geometry-dot-products--cosine-distance)
2. [Matrix Multiplication & Linear Transformations](#2-matrix-multiplication--linear-transformations)
3. [Determinant as Volume Scaling & Matrix Invertibility](#3-determinant-as-volume-scaling--matrix-invertibility)
4. [Eigenvalues, Eigenvectors & Power Iteration](#4-eigenvalues-eigenvectors--power-iteration)
5. [Singular Value Decomposition (SVD) & Low-Rank Compression](#5-singular-value-decomposition-svd--low-rank-compression)
6. [Principal Component Analysis (PCA) from Scratch](#6-principal-component-analysis-pca-from-scratch)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"NumPy Version: {np.__version__}")

## 1. Vector Geometry, Dot Products & Cosine Distance
The dot product between vectors $u, v \in \mathbb{R}^D$ is:
$$\langle u, v \rangle = u^T v = \sum_{i=1}^D u_i v_i = \|u\|_2 \|v\|_2 \cos(\theta)$$

In [ ]:
# Define two sample feature vectors
u = np.array([3.0, 4.0])
v = np.array([4.0, 1.0])

dot_prod = np.dot(u, v)
norm_u = np.linalg.norm(u)
norm_v = np.linalg.norm(v)
cos_theta = dot_prod / (norm_u * norm_v)
angle_deg = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))

# Vector Projection of u onto v: proj_v(u) = (u . v / ||v||^2) * v
proj_v_u = (dot_prod / (norm_v ** 2)) * v

print(f"Dot Product:      {dot_prod:.2f}")
print(f"Cosine Similarity: {cos_theta:.4f}")
print(f"Angle (degrees):   {angle_deg:.2f}°")
print(f"Projection of u on v: {proj_v_u}")

## 2. Matrix Multiplication & Linear Transformations
A matrix $A \in \mathbb{R}^{M \times N}$ maps vectors from $\mathbb{R}^N$ to $\mathbb{R}^M$.

In [ ]:
# Unit circle sampling to visualize 2D transformation
theta = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])  # (2, 200)

# Linear transformation matrix: Shear and Scale
A = np.array([[2.0, 1.0], [0.5, 1.5]])
transformed = A @ circle

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(circle[0], circle[1], label="Original Unit Circle", color="#1f77b4")
plt.title("Input Space (Domain)", fontweight="bold")
plt.axis("equal")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(transformed[0], transformed[1], label="Transformed Ellipse", color="#d62728")
plt.title(f"Transformed by Matrix A (det={np.linalg.det(A):.2f})", fontweight="bold")
plt.axis("equal")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Determinant as Volume Scaling & Matrix Invertibility
The determinant $|\det(A)|$ represents the factor by which area (in 2D) or volume (in $N$-D) is scaled under the transformation. If $\det(A) = 0$, space is collapsed into a lower-dimensional subspace, and the matrix is non-invertible (singular).

In [ ]:
# Invertible matrix vs Singular matrix
A_invertible = np.array([[3.0, 1.0], [1.0, 2.0]])
A_singular = np.array([[2.0, 4.0], [1.0, 2.0]])  # Row 1 is 2 * Row 2

det_inv = np.linalg.det(A_invertible)
det_sing = np.linalg.det(A_singular)

print(f"det(A_invertible) = {det_inv:.4f} (Invertible: {not np.isclose(det_inv, 0)})")
print(f"det(A_singular)   = {det_sing:.4f} (Invertible: {not np.isclose(det_sing, 0)})")

# Compute inverse
A_inv = np.linalg.inv(A_invertible)
print("A @ A^{-1} =")
print(np.round(A_invertible @ A_inv, 6))

## 4. Eigenvalues, Eigenvectors & Power Iteration
For a square matrix $A$, eigenvectors $v$ maintain their direction under $A$, being scaled merely by eigenvalue $\lambda$:
$$A v = \lambda v$$

In [ ]:
# Symmetric positive-definite matrix
S = np.array([[5.0, 2.0], [2.0, 3.0]])
eigenvals, eigenvecs = np.linalg.eigh(S)

# Implementing Power Iteration to extract the dominant eigenvector
b_k = np.random.randn(2)
b_k = b_k / np.linalg.norm(b_k)
for _ in range(50):
    b_next = S @ b_k
    b_k = b_next / np.linalg.norm(b_next)

dominant_eigenvalue = float(b_k.T @ S @ b_k)
print(f"Exact Largest Eigenvalue:   {eigenvals[-1]:.6f}")
print(f"Power Iteration Estimate:   {dominant_eigenvalue:.6f}")
print(f"Dominant Eigenvector:       {b_k}")

## 5. Singular Value Decomposition (SVD) & Low-Rank Compression
Any real matrix $A \in \mathbb{R}^{M \times N}$ decomposes as:
$$A = U \Sigma V^T$$
By the Eckart-Young-Mirsky theorem, truncating to rank-$k$ gives the optimal Frobenius-norm approximation $A_k = \sum_{i=1}^k \sigma_i u_i v_i^T$.

In [ ]:
# Create synthetic low-rank signal corrupted with noise
np.random.seed(42)
rank_true = 3
A_clean = np.random.randn(50, rank_true) @ np.random.randn(rank_true, 40)
A_noisy = A_clean + 0.1 * np.random.randn(50, 40)

U, S, Vt = np.linalg.svd(A_noisy, full_matrices=False)

# Reconstruct using rank-k truncation
k = 3
A_reconstructed = (U[:, :k] * S[:k]) @ Vt[:k, :]

error_noisy = np.linalg.norm(A_noisy - A_clean, 'fro')
error_denoised = np.linalg.norm(A_reconstructed - A_clean, 'fro')
print(f"Frobenius Error (Noisy Matrix):      {error_noisy:.4f}")
print(f"Frobenius Error (Denoised Rank-{k}):   {error_denoised:.4f}")
assert error_denoised < error_noisy

## 6. Principal Component Analysis (PCA) from Scratch
PCA projects centered data onto the top eigenvectors of the sample covariance matrix $\Sigma = \frac{1}{N-1} X_c^T X_c$, which are identical to the right singular vectors $V$ of centered data $X_c$.

In [ ]:
# Generate correlated 3D data
np.random.seed(0)
n_samples = 300
x1 = np.random.normal(0, 3, size=n_samples)
x2 = 0.8 * x1 + np.random.normal(0, 1, size=n_samples)
x3 = 0.3 * x1 - 0.5 * x2 + np.random.normal(0, 0.5, size=n_samples)
X = np.column_stack([x1, x2, x3])

# Step 1: Center data
mu = np.mean(X, axis=0, keepdims=True)
X_centered = X - mu

# Step 2: SVD
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Explained variance ratios
eigenvals = (S ** 2) / (n_samples - 1)
var_ratios = eigenvals / np.sum(eigenvals)
print(f"Explained Variance Ratios: {np.round(var_ratios, 4)}")

# Step 3: Project onto 2 principal components
W2 = Vt[:2].T  # (3, 2)
X_pca = X_centered @ W2  # (300, 2)

plt.figure(figsize=(7, 4.5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c="#2b5c8f", alpha=0.7, edgecolors="none")
plt.title(f"PCA 2D Projection (Retaining {np.sum(var_ratios[:2])*100:.1f}% Variance)", fontweight="bold")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.tight_layout()
plt.show()